# Generating surface plots of a single cortical hemisphere

This notebook is a tutorial for generating visualisations of a single cortical surface hemisphere using ``plotting.plot_surf``.

In [ ]:
from importlib.resources import files
import matplotlib.pyplot as plt
import plotly.io as pio
import nibabel as nib
from neuromodes.io import fetch_surf
from neuromodes import EigenSolver
from nsbutils.utils import unmask
from nsbutils.plotting import plot_surf

pio.renderers.default = "sphinx_gallery"

We start by loading the necessary surfaces, and medial masks.

In [ ]:
species = 'human'
den = "32k"

surf_lh_data, medmask_lh = fetch_surf(density=den, hemi="L", species=species)
surf_rh_data, medmask_rh = fetch_surf(density=den, hemi="R", species=species)
surf_lh = {"vertices": surf_lh_data.v, "faces": surf_lh_data.t}
surf_rh = {"vertices": surf_rh_data.v, "faces": surf_rh_data.t}

We can calculate geometrice eigenmodes to use as data to plot on the surfaces.

In [ ]:
solver_lh = EigenSolver(surf_lh, medmask_lh)
solver_lh.solve(n_modes=100)

modes_unmasked = unmask(solver_lh.emodes, medmask_lh)

Now we can plot the first non-constant eigenmode as an interactive plot. If you spin the surface around you'll notice that the medial wall is grey. This is handled by `rois` where any vertex labelled with `0` is greyed-out.

In [ ]:
surfs = {"lh": surf_lh}
data = {"lh": modes_unmasked[:, 1]}
rois = {"lh": medmask_lh}

fig = plot_surf(
    surf=surfs, data=data, rois=rois, views=["lateral"], cmap="RdBu"
)
fig.show()

We can optionally plot the mesh edges, add a colobar, and increase the figure size.

In [ ]:
fig = plot_surf(
    surf=surfs, data=data, rois=rois, views=["lateral"], cmap="RdBu", cbar=True, 
    mesh_edges=True, size=(700,400)
)
fig.show()

We can also plot multiple views. Note that the order of the views is determined by how they are ordered in `views`.

In [ ]:
fig = plot_surf(
    surf=surfs, data=data, rois=rois, views=["lateral", "medial"], cmap="RdBu",
)
fig.show()

We can also plot a parcellation by inputting it into `rois` and setting `roi_outline=True`. Zeros will again be masked out.

In [ ]:
parc_lh_data = files('nsbutils').parent / 'docs' / 'tutorials' / 'tutorial_data' / 'Q1-Q6_RelatedParcellation210.L.CorticalAreas_dil_Colors.32k_fs_LR.label.gii'
parc_lh = nib.load(parc_lh_data).darrays[0].data.astype(int)
rois_parc = {"lh": parc_lh}

fig = plot_surf(
    surf=surfs, data=data, rois=rois_parc, views=["lateral"], cmap="RdBu", roi_outlines=True, size=(700,400)
)
fig.show()

We can plot multiple maps by redifining `surfs["lh"]` to be a 2D array. Note that `size` adjusts the size of each subplot.

In [ ]:
mode_ids = [1,2,3,99]
data_multi = {"lh": modes_unmasked[:, mode_ids]}

fig = plot_surf(
    surf=surfs, data=data_multi, rois=rois, views=["lateral"], cmap="RdBu", size=(250, 150)
)
fig.show()

`plot_surf` always returns a plotly figure but we can optionally pass in a matplotlib axis using the `ax` argument in which case the figure will be converted to a static image and inserted into the axis. We can use the `scale` argument to increase the resolution.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

data = {
    "lh": unmask(solver_lh.emodes[:, mode_ids], medmask_lh), 
}
fig_plotly = plot_surf(
    surf=surfs, data=data, rois=rois, views=["lateral"], 
    layout_indiv="grid", cmap="RdBu", ax=ax, scale=2.0
)

plt.show()